# Circle

A disc of a given radius: the cross-section of a wire, a rod or a round electrode.

`Circle(midpoint, radius)` has an exact level set, the signed distance
`|p - midpoint| - radius`, so distances, closest points and normals are analytic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Circle, Grid

rod = Circle(midpoint=(0.0, 0.0), radius=0.5)
points = np.array([[0.0, 0.4, 1.0], [0.0, 0.0, 1.0]])  # (2, n): one point per column
print("inside:", rod.contains_points(points))
print("signed distance:", rod.signed_distance(points))
print("closest point to (1, 1):", rod.closest_point((1.0, 1.0)))
print("normal at (0.5, 0):", rod.surface_normal((0.5, 0.0)))
assert np.isclose(rod.signed_distance((1.0, 1.0)), np.sqrt(2) - 0.5)

## Distance field and node mask

The level set sampled around the rod: its contours are circles, one per distance. The grid
nodes inside the rod are those `node_mask` selects.

In [ ]:
x = np.linspace(-1.0, 1.0, 201)
X, Y = np.meshgrid(x, x, indexing="ij")
distance = rod.signed_distance(np.stack((X.ravel(), Y.ravel()))).reshape(X.shape)

grid = Grid(2, (-1.0, -1.0), (1.0, 1.0), (21, 21))
mask = rod.node_mask(grid).ravel()
nodes = grid.node_points()

fig, ax = plt.subplots(figsize=(5.5, 4.5), layout="constrained")
image = ax.contourf(X, Y, distance, levels=np.linspace(-0.5, 1.0, 16), cmap="RdBu_r")
ax.contour(X, Y, distance, levels=[0.0], colors="k")
ax.scatter(*nodes[:, mask], s=8, color="k", label="nodes inside")
fig.colorbar(image, ax=ax, label="signed distance")
ax.set(xlabel="x", ylabel="y", title="The level set of a Circle")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.show()

## Outlines and inversion

`outlines(max_spacing)` gives the boundary as a counter-clockwise vertex loop, for
boundary-element solvers and for plotting. An inverted circle is everything outside it: the
wall of a round chamber.

In [ ]:
(loop,) = rod.outlines(max_spacing=0.05)
print("outline vertices:", loop.shape[1])

chamber = Circle(midpoint=(0.0, 0.0), radius=0.5, inverted=True)
assert np.array_equal(chamber.node_mask(grid).ravel(), ~mask)
assert np.isclose(chamber.signed_distance((1.0, 1.0)), -rod.signed_distance((1.0, 1.0)))